# Suite CNT — The container layout

`Container` gives each container shape a grid of cells, and places the children it gains and closes the gaps it loses, one change at a time, with no coordinates of any kind: mbse-basis [CAP-238: container schemas and grid insertion](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-238), by the rules of [the diagrams proposal](https://github.com/pitaman71/mbse-basis/blob/main/proposals/DIAGRAMS.md).

In [ ]:
import { Builders, Validators } from "@mbse/schemas/Framework";

import { Container, Logical } from "../src/Framework/index.js";

function assert(condition: unknown, message = "assertion failed"): asserts condition {
  if (!condition) throw new Error(`AssertionError: ${message}`);
}

const NAMES = ["Container.Layout", "Container.OfGrid", "Container.Cells", "Container.OfCell", "Container.Holds",
  "Logical.OfShape"];
const store: any = new Builders.Store();
store.transaction(Logical.install);
store.transaction(Container.install);
const L = store.Logical;
const diagrams = store.singleton("Logical.Diagrams");
const problems = (schema: any, root: any): string[] => Validators.Validate(store).Reachable(schema, root);
const text = (value: unknown): string => JSON.stringify(value, (_k, v) => typeof v === "bigint" ? Number(v) : v);

store.transaction(() =>
  L.DiagramDirectory(diagrams, (f: any) => f.diagrams((a: any) => a.insert((e: any) => e.name("overview").diagram((d: any) => d)))));
const overview = diagrams.diagrams({ name: "overview" });

/** A top-level shape of the diagram, holding a child of each name in `children`. */
function box(name: string, children = ""): any {
  const spec = (s: any): any => {
    for (const child of children) s.children((k: any) => k.insert((e: any) => e.name(child).child((c: any) => c)));
    return s;
  };
  L.OfDiagram(overview, (d: any) => d.shapes((s: any) => s.insert((e: any) => e.name(name).shape(spec))));
  return overview.shapes({ name });
}

function add(shape: any, name: string): void {
  L.OfShape(shape, (s: any) => s.children((k: any) => k.insert((e: any) => e.name(name).child((c: any) => c))));
  Container.added(shape.children({ name }));
}

function remove(shape: any, name: string): void {
  L.OfShape(shape, (s: any) => s.children((k: any) => k.remove((e: any) => e.name(name))));
  Container.removed(shape);
}

/** Each child's name and cell, in order, and the grid's extents, as JSON text. */
function cells(shape: any): string {
  const [entry] = [...shape.grid];
  const grid = entry.grid;
  return text([[...shape.children].map((e: any) => [e.name.get(), Container.cell_of(e.child)]),
    [grid.rows.get(), grid.columns.get()]]);
}

function raises(message: string, block: () => unknown): void {
  try {
    store.transaction(block);
  } catch (error) {
    assert(error instanceof Error && error.name === "ValueError", String(error));
    assert(error.message === message, error.message);
    return;
  }
  throw new Error(`AssertionError: expected ValueError: ${message}`);
}

<a id="CNT-01"></a>
## CNT-01 · The install, after Logical's, defines the container layout's schemas, each valid, and run again writes nothing

In [ ]:
const schemas = NAMES.map((name) => store.registered(name));
assert(schemas.every((schema: any) => schema.validate().length === 0));
store.transaction(Container.install);
assert(NAMES.every((name, i) => store.registered(name) === schemas[i]));
const adjacencies = store.registered("Logical.OfShape").adjacencies;
assert(adjacencies.has("grid") && adjacencies.has("cell"));

<a id="CNT-02"></a>
## CNT-02 · A grid placed on a container holds the children it already has, in order, each in its own cell

In [ ]:
let cpu: any, grid: any;
store.transaction(() => {
  cpu = box("cpu", "abc");
  grid = Container.layout(cpu, 2n);
});
assert([...cpu.grid][0].grid === grid && [...grid.shape][0].shape === cpu);
assert(grid.grows.get() === "rows" && grid.span.get() === 2n);
assert(cells(cpu) === text([[["a", [0, 0]], ["b", [0, 1]], ["c", [1, 0]]], [2, 2]]));
assert(Container.cell_of(cpu) === null && problems(L.OfShape, cpu).length === 0);

<a id="CNT-03"></a>
## CNT-03 · A child added takes the next cell, and the grid's extents grow

In [ ]:
store.transaction(() => add(cpu, "d"));
assert(cells(cpu) === text([[["a", [0, 0]], ["b", [0, 1]], ["c", [1, 0]], ["d", [1, 1]]], [2, 2]]));
store.transaction(() => add(cpu, "e"));
assert(cells(cpu) === text([[["a", [0, 0]], ["b", [0, 1]], ["c", [1, 0]], ["d", [1, 1]], ["e", [2, 0]]], [3, 2]]));
let one: any;
store.transaction(() => {
  one = box("one");
  Container.layout(one, 3n);
});
assert(cells(one) === text([[], [0, 0]]));
store.transaction(() => add(one, "x"));
assert(cells(one) === text([[["x", [0, 0]]], [1, 1]]) && problems(L.OfShape, cpu).length === 0);

<a id="CNT-04"></a>
## CNT-04 · A grid that grows columns fills each column first

In [ ]:
let bus: any;
store.transaction(() => {
  bus = box("bus", "abcde");
  Container.layout(bus, 2n, "columns");
});
assert(cells(bus) === text([[["a", [0, 0]], ["b", [1, 0]], ["c", [0, 1]], ["d", [1, 1]], ["e", [0, 2]]], [2, 3]]));

<a id="CNT-05"></a>
## CNT-05 · A child removed leaves a gap the children after it close, one cell back each

In [ ]:
store.transaction(() => remove(cpu, "b"));
assert(cells(cpu) === text([[["a", [0, 0]], ["c", [0, 1]], ["d", [1, 0]], ["e", [1, 1]]], [2, 2]]));
store.transaction(() => remove(cpu, "e"));
assert(cells(cpu) === text([[["a", [0, 0]], ["c", [0, 1]], ["d", [1, 0]]], [2, 2]]));
store.transaction(() => {
  remove(bus, "a");
  remove(bus, "d");
});
assert(cells(bus) === text([[["b", [0, 0]], ["c", [1, 0]], ["e", [0, 1]]], [2, 2]]));
store.transaction(() => remove(one, "x"));
assert(cells(one) === text([[], [0, 0]]) && problems(L.OfShape, cpu).length === 0 && problems(L.OfShape, bus).length === 0);

<a id="CNT-06"></a>
## CNT-06 · What a grid refuses

In [ ]:
raises("a grid grows rows or columns, not 'diagonal'", () => Container.layout(box("diagonal"), 2n, "diagonal"));
raises("a grid's span is a positive int, not 0", () => Container.layout(box("empty"), 0n));
raises("a shape has one grid", () => Container.layout(cpu, 2n));
raises("a container without a grid has no container layout: give it one, with layout", () => add(box("bare"), "y"));